# Director-AI — LangChain Integration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/07_langchain_integration.ipynb)

How to use Director-AI with LangChain chains via the `CoherenceCallbackHandler`.

Covers:
1. Callback handler setup
2. Scoring LangChain chain outputs
3. Manual scorer integration

In [ ]:
!pip install -q director-ai

## 1. CoherenceCallbackHandler

The `CoherenceCallbackHandler` from `director_ai.integrations.langchain_callback`
hooks into LangChain's callback system and scores each LLM output.

In [ ]:
from director_ai.core import CoherenceScorer, GroundTruthStore

store = GroundTruthStore()
store.add("python", "Python was created by Guido van Rossum and first released in 1991.")
store.add("java", "Java was developed by James Gosling at Sun Microsystems, released in 1995.")

scorer = CoherenceScorer(
    threshold=0.6,
    ground_truth_store=store,
    use_nli=False,
)

print("Scorer and knowledge base ready")

## 2. Direct Scoring (No LangChain Dependency)

Score any LLM output manually — works with any framework.

In [ ]:
# Simulate chain outputs
chain_outputs = [
    {"query": "Who created Python?", "result": "Python was created by Guido van Rossum."},
    {"query": "Who created Python?", "result": "Python was created by Linus Torvalds in 2005."},
    {"query": "When was Java released?", "result": "Java was released in 1995 by Sun Microsystems."},
]

for output in chain_outputs:
    approved, score = scorer.review(output["query"], output["result"])
    status = "PASS" if approved else "BLOCKED"
    print(f"[{status}] {output['result']}")
    print(f"  coherence={score.score:.3f}\n")

## 3. Callback Handler Pattern

If you have `langchain-core` installed, the callback handler integrates directly.
Here we show the pattern without requiring langchain as a dependency.

In [ ]:
# Pattern for LangChain callback integration:
#
#   from director_ai.integrations.langchain_callback import CoherenceCallbackHandler
#   handler = CoherenceCallbackHandler(scorer=scorer)
#   chain = LLMChain(llm=llm, prompt=prompt, callbacks=[handler])
#   result = chain.invoke({"question": "Who created Python?"})
#
# The handler scores every LLM output and logs warnings for low coherence.

# Simulated callback flow:
class SimpleCallbackDemo:
    def __init__(self, scorer):
        self.scorer = scorer
        self.scores = []

    def on_llm_end(self, prompt, response_text):
        approved, score = self.scorer.review(prompt, response_text)
        self.scores.append((approved, score))
        if not approved:
            print(f"  WARNING: low coherence ({score.score:.3f}) on: {response_text[:60]}")
        return approved

cb = SimpleCallbackDemo(scorer)
cb.on_llm_end("Who created Python?", "Python was created by Guido van Rossum.")
cb.on_llm_end("Who created Python?", "Python was invented by Steve Jobs.")

print(f"\nScored {len(cb.scores)} outputs")
for approved, sc in cb.scores:
    print(f"  approved={approved}  score={sc.score:.3f}")

## Summary

- Director-AI integrates with LangChain via callback handlers
- Any chain output can be scored with `scorer.review(prompt, response)`
- The scorer works independently of any framework — use it anywhere